# Lewisham · `borough` and `lpa_name` (web: "Borough"; lpa_name is not a web column)

**Goal:** understand the data, so that a chat bot can answer any question about it
more easily and correctly. Lewisham also serves as a check on what we found for Brent.

Lewisham only (golden rule): every query carries the location-based Lewisham filter.
Data comes from the **live API** (the bot's only source), so counts change as the
Datahub is updated.

Together these two fields make the scope filter: borough is Lewisham, or borough is
missing and lpa_name is Lewisham.

Workflow: add a step here → discuss the findings → only after agreement update
`semantic/lewisham/` and `knowledge/`.

In [1]:
from pathlib import Path

import httpx
import pandas as pd
from dotenv import dotenv_values

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
env = dotenv_values(PROJECT_ROOT / ".env")

BASE_URL = "https://planningdata.london.gov.uk/api-guest"
HEADERS = {"X-API-AllowRequest": env["DATAHUB_ALLOW_REQUEST"]}


def lewisham_filter() -> dict:
    # Location-based scope: borough is Lewisham, or borough is missing and Lewisham decided it.
    return {
        "bool": {
            "should": [
                {"term": {"borough.raw": "Lewisham"}},
                {"bool": {
                    "filter": [{"term": {"lpa_name.raw": "Lewisham"}}],
                    "must_not": [{"exists": {"field": "borough.raw"}}],
                }},
            ],
            "minimum_should_match": 1,
        }
    }


def search(body: dict) -> dict:
    response = httpx.post(f"{BASE_URL}/applications/_search", headers=HEADERS, json=body, timeout=60)
    response.raise_for_status()
    return response.json()

## Step 1 · Value counts: `borough` × `lpa_name`

1. **In scope:** every Lewisham record, by `borough` and `lpa_name` (`missing` shown as
   "(none)"), via aggregations.
2. **Just outside scope** (count only, read-only EDA exception approved 2026-11-11):
   records decided by Lewisham (`lpa_name` Lewisham) whose `borough` is another one, by
   borough. The filter leaves these out; Brent had 2 the other way round (decided by
   Hammersmith & Fulham, borough Brent).

In [2]:
body = {"size": 0, "track_total_hits": True, "query": lewisham_filter(),
        "aggs": {"borough": {"terms": {"field": "borough.raw", "size": 50, "missing": "(none)"},
                             "aggs": {"lpa": {"terms": {"field": "lpa_name.raw", "size": 50, "missing": "(none)"}}}}}}
data = search(body)
print("Lewisham records:", data["hits"]["total"]["value"])
pd.DataFrame([(b["key"], l["key"], l["doc_count"]) for b in data["aggregations"]["borough"]["buckets"]
              for l in b["lpa"]["buckets"]], columns=["borough", "lpa_name", "count"])

Lewisham records: 29021


,borough,lpa_name,count
0,Lewisham,Lewisham,28569
1,(none),Lewisham,452


In [3]:
# Just outside scope: decided by Lewisham, borough another one (counts only).
body = {"size": 0, "query": {"bool": {"filter": [{"term": {"lpa_name.raw": "Lewisham"}}, {"exists": {"field": "borough.raw"}}],
                                      "must_not": [{"term": {"borough.raw": "Lewisham"}}]}},
        "aggs": {"borough": {"terms": {"field": "borough.raw", "size": 50}}}}
outside = search(body)
print("lpa_name Lewisham, borough not Lewisham:", outside["hits"]["total"]["value"],
      {b["key"]: b["doc_count"] for b in outside["aggregations"]["borough"]["buckets"]})

lpa_name Lewisham, borough not Lewisham: 0 {}


### Findings · Step 1

| borough | lpa_name | Count |
|---|---|---|
| Lewisham | Lewisham | 28,569 |
| (none) | Lewisham | 452 |

- **One authority only:** every Lewisham record has lpa_name Lewisham. No other
  decision-maker (Brent has OPDC for 154 records).
- **452 have no borough, all of them empty records** (status notebook, Step 2: 452 of the
  1,151 empty records have no borough), so they are dropped. **After cleaning, every
  record has borough Lewisham.**
- **No Lewisham-decided record is filed under another borough** (0). Brent had 2 records
  of another authority filed under Brent (Hammersmith & Fulham, dropped).
- **borough is the recording authority, not proof of the site:** the consultation
  responses (out-of-borough notebook) have borough Lewisham with sites elsewhere.

## Step 2 · Type check

Aggregations hide the JSON type, so every record's `borough` and `lpa_name` are fetched
(paged with `search_after`, sorted by `id`) and the Python type counted, with the exact
text (`repr`) to catch case or whitespace variants.

In [4]:
rows, after = [], None
while True:
    body = {"size": 5000, "query": lewisham_filter(), "_source": ["id", "borough", "lpa_name"], "sort": [{"id": "asc"}]}
    if after:
        body["search_after"] = after
    page = search(body)["hits"]["hits"]
    if not page:
        break
    rows += [h["_source"] for h in page]
    after = page[-1]["sort"]

bl = pd.DataFrame(rows).reindex(columns=["id", "borough", "lpa_name"])
for f in ["borough", "lpa_name"]:
    print(f, "types:", bl[f].map(lambda v: "null" if v is None or v != v else type(v).__name__).value_counts().to_dict(),
          "| exact values:", bl[f].dropna().map(repr).value_counts().to_dict())

borough types: {'str': 28569, 'null': 452} | exact values: {"'Lewisham'": 28569}
lpa_name types: {'str': 29021} | exact values: {"'Lewisham'": 29021}


### Findings · Step 2

- **Strings or null only:** borough 28,569 strings, 452 null; lpa_name 29,021 strings.
- **One exact value each:** `'Lewisham'`; no case or whitespace variants.

## Step 3 · Does any other borough or authority name contain "Lewisham"?

The scope filter matches `borough` and `lpa_name` exactly. A value such as "Lewisham
Borough" or "LB Lewisham" would be missed. So every distinct `borough` and `lpa_name` in
the whole Datahub is listed (names and counts only; read-only EDA exception approved
2026-11-11), and those containing "lewisham" (case-insensitive) are shown. Brent's check
found "Market Quarter, Brent Cross South" (a Barnet site).

In [5]:
body = {"size": 0, "aggs": {"borough": {"terms": {"field": "borough.raw", "size": 1000}},
                             "lpa_name": {"terms": {"field": "lpa_name.raw", "size": 1000}}}}
aggs = search(body)["aggregations"]
for f in ["borough", "lpa_name"]:
    names = [(b["key"], b["doc_count"]) for b in aggs[f]["buckets"]]
    print(f"{f}: {len(names)} distinct | containing 'lewisham':", [n for n in names if "lewisham" in n[0].lower()])

borough: 155 distinct | containing 'lewisham': [('Lewisham', 28569)]
lpa_name: 36 distinct | containing 'lewisham': [('Lewisham', 29021)]


### Findings · Step 3

- **Only the exact name:** of 155 distinct `borough` values and 36 distinct `lpa_name`
  values in the whole Datahub, the only one containing "lewisham" is `Lewisham` itself
  (borough 28,569, lpa_name 29,021). No variant is missed by the scope filter.
- **Every Lewisham-decided record is in scope:** lpa_name Lewisham has 29,021 records in
  the whole Datahub, the same as the Lewisham scope count.